# 第13課 - 使用Cognee知識圖譜的代理記憶


## 設定

此筆記本示範如何使用 [**Cognee**](https://www.cognee.ai/) 知識圖譜和 **Microsoft Agent Framework** (MAF) 建立具備持久記憶的智能**編程助理**。

Cognee 將非結構化文本轉換為結構化且可查詢的知識圖譜，並以向量嵌入作為支持，為您的代理提供豐富且具關係意識的長期記憶。

### 您將學到的內容
1. **建立知識圖譜**：將開發者檔案和最佳實踐轉換為結構化且可查詢的知識。
2. **整合 Cognee 與 MAF**：使用 `@tool` 函數讓 MAF 代理查詢 Cognee 的知識圖譜。
3. **支援會話的對話**：在同一會話中保持多個問題的上下文。
4. **長期記憶**：跨會話持久化重要知識，並在新對話中檢索。

### 先決條件
- Python 3.9+
- 本地運行 Redis（`docker run -d -p 6379:6379 redis`）以進行會話管理
- 一個 LLM API 密鑰（例如 OpenAI）— 在 `.env` 中設置 `LLM_API_KEY`
- `.env` 中的 `CACHING=true`（Cognee 會話必需）
- 一個部署了聊天模型的 Azure AI Foundry 項目
- `.env` 中的 `AZURE_AI_PROJECT_ENDPOINT` 和 `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- 已驗證的 Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## 代理記憶類型

這個筆記本探索了與主課程 13 筆記本相同的三種記憶類型，但使用 Cognee 作為長期記憶後端：

| 記憶類型 | 機制 | 壽命 |
|---|---|---|
| **工作記憶** | `agent.create_session()` (MAF) | 單次對話 |
| **短期記憶** | Cognee 會話快取 (Redis) | 單次會話 |
| **長期記憶** | Cognee 知識圖譜 + 向量 | 永久 |

### Cognee 的記憶架構
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## 準備 Cognee 儲存裝置


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Part 1 — 建立知識庫

我們吸收三種類型的數據，以建立一個全面的編碼助手知識庫：

1. **開發人員檔案** — 個人專業知識和技術背景  
2. **Python 最佳實踐** — Python 禪意及實用指引  
3. **歷史對話** — 開發人員與 AI 助手之間的過去問答記錄


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## 視覺化知識圖譜

Cognee 可以呈現其擷取的實體與關係的互動式 HTML 視覺化。


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## 用 Memify 豐富記憶

`memify()` 會分析知識圖譜並生成智能規則 — 識別模式、最佳實踐以及概念間的關係。


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Part 2 — 使用 Cognee 工具的 MAF 代理

現在我們建立一個 MAF 代理，可以透過 `@tool` 函數查詢 Cognee 的知識圖譜。這讓代理能夠利用具圖譜感知的語義搜尋的全部威力，同時通過會話保持對話上下文。


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## 使用 Session 的工作記憶

`AgentSession`（通過 `agent.create_session()` 創建）在一個會話中提供工作記憶。代理人可以回顧早前的訊息，同時查詢 Cognee 的長期知識圖譜。


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## 新對話 — 長期記憶持續存在

開始新的對話會清除工作記憶，但 Cognee 知識圖譜仍然可用。代理可以在全新的對話中檢索相同的長期知識。


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Summary

在這個筆記本中，你建立了一個結合了 **MAF 的工作記憶**（`agent.create_session()`）與 **Cognee 的長期知識圖譜** 的編碼助理。

### 你學到了什麼
1. **知識圖譜構建**：Cognee 解析非結構化文本並建立圖譜＋向量記憶。
2. **使用 memify 進行圖譜增強**：在現有圖譜上產生推導事實和更豐富的關係。
3. **MAF + Cognee 整合**：`@tool` 函數讓 MAF 代理能自然地查詢 Cognee 的圖譜。
4. **工作記憶 + 長期記憶**：`AgentSession`（透過 `agent.create_session()`）提供會話上下文，Cognee 則提供持久知識。
5. **使用 NodeSets 篩選搜索**：針對知識圖中特定子集進行搜尋（例如僅限原則部分）。

### 主要收穫
- **Cognee** 將原始文本轉成結構化、具關係感知的記憶，功能比平面向量庫更強大。
- **`@tool` 函數** 清晰地搭起了 MAF 代理與外部知識系統的橋樑。
- **`AgentSession`**（透過 `agent.create_session()`）保持每次對話的上下文，與長期知識分開管理。
- 同一個知識圖譜可服務多個會話和代理。

### 實際應用場景
- **開發者副駕駛**：程式碼審查、事件分析、架構助理
- **面向客戶的副駕駛**：針對產品文件、常見問題和 CRM 筆記的支援代理
- **內部專家副駕駛**：政策、法律或安全助理針對指引進行推理
- **統一數據層**：將結構化與非結構化數據整合到一個可查詢的圖譜中

### 下一步
- 在 Cognee 中實驗時間意識功能
- 為領域特定圖譜品質定義 OWL 本體
- 新增用戶反饋回路以隨時間優化檢索
- 擴展至共享同一 Cognee 記憶層的多代理系統


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：  
本文件乃使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們致力於確保準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始語言的文件應視為權威來源。對於重要資訊，建議聘請專業人工翻譯。我們對因使用本翻譯而引起的任何誤解或曲解概不負責。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
